In [2]:
# %% 1) Mount Drive + installs + imports + setup
from google.colab import drive
drive.mount('/content/drive')

!pip install -q albumentations openpyxl

import os, random, warnings, cv2, math, time, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from albumentations import Compose, Resize, ShiftScaleRotate, RandomBrightnessContrast, GaussianBlur, CoarseDropout, Lambda, Normalize
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Mounted at /content/drive
Device: cuda


In [3]:
# %% 2) Hyper-params & paths (same structure you used)
SHAPE_SOURCES = {
    "shape7":  ("/content/shape7_labels_all.csv", "/content/drive/MyDrive/shapes/shape7"),
    "shape9":  ("/content/shape9_scores.csv",     "/content/drive/MyDrive/shapes/shape9"),
    "shape14": ("/content/shape14_scores.xlsx",   "/content/drive/MyDrive/shapes/shape14"),
    "shape18": ("/content/shape18_scores.csv",    "/content/drive/MyDrive/shapes/shape18"),
}

IMG_SIZE   = 128
BATCH_SIZE = 32
EPOCHS     = 60
LR         = 3e-4
PATIENCE   = 10
EMB_DIM    = 128        # embedding size for Siamese backbone
TOL        = 1          # ±1 tolerance for accuracy

# Normalization expected by ResNet (single channel)
NORM_MEAN = [0.485]
NORM_STD  = [0.229]


In [4]:
# %% 3) Smart auto-crop (same behavior)
def smart_center_crop(img: np.ndarray, thr=20):
    ys, xs = np.where(img > thr)
    if xs.size == 0:
        return img
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    side = max(x1 - x0, y1 - y0) + 2
    cx, cy = (x0 + x1)//2, (y0 + y1)//2
    half = side//2
    x0, x1, y0, y1 = cx-half, cx+half, cy-half, cy+half
    pad_x0, pad_y0 = max(-x0,0), max(-y0,0)
    pad_x1, pad_y1 = max(x1-img.shape[1]+1,0), max(y1-img.shape[0]+1,0)
    if pad_x0+pad_x1+pad_y0+pad_y1:
        img = np.pad(img, ((pad_y0,pad_y1),(pad_x0,pad_x1)), constant_values=0)
        y0+=pad_y0; y1+=pad_y0; x0+=pad_x0; x1+=pad_x0
    return img[y0:y1+1, x0:x1+1]

class SmartCrop:
    def __init__(self, thr=20): self.thr=thr
    def __call__(self, img, **kw): return smart_center_crop(img, self.thr)


In [5]:
# %% 4) Transforms (augment train, normalize both)
train_tf = Compose([
    Lambda(image=SmartCrop(20)),
    Resize(IMG_SIZE, IMG_SIZE),
    ShiftScaleRotate(0.05, 0.05, 10, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.8),
    RandomBrightnessContrast(0.2, 0.2, p=0.5),
    GaussianBlur(blur_limit=3, p=0.3),
    CoarseDropout(max_holes=4, max_height=16, max_width=16,
                  min_holes=1, min_height=8, min_width=8,
                  fill_value=0, p=0.3),
    Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2()
])
val_tf = Compose([
    Lambda(image=SmartCrop(20)),
    Resize(IMG_SIZE, IMG_SIZE),
    Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2()
])


In [6]:

# %% 5) Build merged dataframe and resolve reference images
def load_all_shapes(shape_sources):
    frames = []
    for shape_name, (score_file, img_dir) in shape_sources.items():
        if score_file.lower().endswith((".xls", ".xlsx")):
            df_shape = pd.read_excel(score_file)
        else:
            df_shape = pd.read_csv(score_file, encoding="utf-8")
        df_shape["img_path"] = df_shape["child_id"].apply(lambda cid: os.path.join(img_dir, f"{cid}.png"))
        df_shape = df_shape[df_shape["img_path"].apply(os.path.exists)]
        df_shape["shape_name"] = shape_name
        df_shape["img_dir"] = img_dir
        frames.append(df_shape)
    return pd.concat(frames, ignore_index=True)

def find_reference_in_dir(img_dir):
    # Prefer exact "originalX.png" if present; otherwise first file starting with "original"
    cand = [p for p in glob.glob(os.path.join(img_dir, "original*.png"))]
    return cand[0] if len(cand) else None

df = load_all_shapes(SHAPE_SOURCES)
df["ref_path"] = df["img_dir"].apply(find_reference_in_dir)
missing_ref = df["ref_path"].isna().sum()
if missing_ref:
    print(f"⚠️ Could not find reference for {missing_ref} rows. These will be dropped.")
df = df.dropna(subset=["ref_path"]).reset_index(drop=True)

print("Total usable samples:", len(df))
df.head()


Total usable samples: 3557


,child_id,label,img_path,shape_name,img_dir,ref_path
0,4641,1,/content/drive/MyDrive/shapes/shape7/4641.png,shape7,/content/drive/MyDrive/shapes/shape7,/content/drive/MyDrive/shapes/shape7/original7...
1,7501,1,/content/drive/MyDrive/shapes/shape7/7501.png,shape7,/content/drive/MyDrive/shapes/shape7,/content/drive/MyDrive/shapes/shape7/original7...
2,7502,2,/content/drive/MyDrive/shapes/shape7/7502.png,shape7,/content/drive/MyDrive/shapes/shape7,/content/drive/MyDrive/shapes/shape7/original7...
3,7504,3,/content/drive/MyDrive/shapes/shape7/7504.png,shape7,/content/drive/MyDrive/shapes/shape7,/content/drive/MyDrive/shapes/shape7/original7...
4,7505,4,/content/drive/MyDrive/shapes/shape7/7505.png,shape7,/content/drive/MyDrive/shapes/shape7,/content/drive/MyDrive/shapes/shape7/original7...


In [7]:
# %% 6) Siamese Dataset (returns child, reference, label)
class SiameseShapeDataset(Dataset):
    def __init__(self, df, tf_child, tf_ref):
        self.df = df.reset_index(drop=True)
        self.tf_child = tf_child
        self.tf_ref = tf_ref

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # child image
        img_c = cv2.imread(row.img_path, cv2.IMREAD_GRAYSCALE)
        if img_c is None:
            img_c = np.zeros((200,200), np.uint8)
        img_c = self.tf_child(image=img_c)["image"].float()

        # reference image (same folder)
        img_r = cv2.imread(row.ref_path, cv2.IMREAD_GRAYSCALE)
        if img_r is None:
            img_r = np.zeros((200,200), np.uint8)
        img_r = self.tf_ref(image=img_r)["image"].float()

        y = torch.tensor(row.label, dtype=torch.float32)
        return img_c, img_r, y


In [8]:
# %% 7) Siamese Model (shared ResNet18 backbone -> embeddings -> regression head)
def make_backbone(embed_dim=EMB_DIM, trainable_layers=("layer4", "fc")):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)  # grayscale
    # remove final FC, keep global pooling
    in_feats = m.fc.in_features
    m.fc = nn.Identity()
    # projection to embedding
    proj = nn.Linear(in_feats, embed_dim)

    # freeze except some layers
    for n, p in m.named_parameters():
        p.requires_grad = any(n.startswith(tl) for tl in trainable_layers)
    return m, proj

class SiameseRegressor(nn.Module):
    def __init__(self, embed_dim=EMB_DIM):
        super().__init__()
        self.backbone, self.proj = make_backbone(embed_dim)
        # fusion of (f1, f2, |f1-f2|, f1*f2)
        fusion_in = embed_dim*4
        self.head = nn.Sequential(
            nn.Linear(fusion_in, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1)
        )

    def encode(self, x):
        z = self.backbone(x)
        z = self.proj(z)
        return F.normalize(z, p=2, dim=1)

    def forward(self, x1, x2):
        f1 = self.encode(x1)
        f2 = self.encode(x2)
        f_abs = torch.abs(f1 - f2)
        f_mul = f1 * f2
        fused = torch.cat([f1, f2, f_abs, f_mul], dim=1)
        raw = self.head(fused).squeeze(1)
        # bound to [1,7]
        return 1.0 + 6.0 * torch.sigmoid(raw)


In [9]:
# %% 8) Metrics
def mae(pred, targ):
    return torch.mean(torch.abs(pred - targ)).item()

def acc_std(pred, targ):
    return (torch.round(pred).int() == targ.int()).float().mean().item() * 100

def acc_tol(pred, targ, tol=TOL):
    return (torch.abs(torch.round(pred) - targ) <= tol).float().mean().item() * 100


In [10]:
# %% 9) Train/validate helpers
def run_epoch(model, loader, opt=None, loss_fn=nn.MSELoss()):
    is_train = opt is not None
    model.train(is_train)
    total_loss = 0.0
    preds, targs = [], []

    for xb_c, xb_r, yb in loader:
        xb_c, xb_r, yb = xb_c.to(device), xb_r.to(device), yb.to(device)

        with torch.set_grad_enabled(is_train):
            out = model(xb_c, xb_r)
            loss = loss_fn(out, yb)
            if is_train:
                opt.zero_grad()
                loss.backward()
                opt.step()

        total_loss += loss.item() * xb_c.size(0)
        preds.append(out.detach().cpu())
        targs.append(yb.detach().cpu())

    preds = torch.cat(preds); targs = torch.cat(targs)
    return (total_loss / len(loader.dataset),
            mae(preds, targs),
            acc_std(preds, targs),
            acc_tol(preds, targs),
            preds, targs)


In [11]:
# %% 10) Data splits + loaders (held-out test, 5-fold CV on train+val)
df_trainval, df_test = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
print(f"Train+Val: {len(df_trainval)} | Test: {len(df_test)}")

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)

def make_loader(sdf, tf, shuffle=False, sampler=None):
    return DataLoader(
        SiameseShapeDataset(sdf, tf, tf),
        batch_size=BATCH_SIZE,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=2,
        pin_memory=True
    )


Train+Val: 2845 | Test: 712


In [12]:
# %% 11) 5-fold training
from tqdm.notebook import tqdm

mae_list, std_list, tol_list = [], [], []
fold_models = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(df_trainval, df_trainval.label), 1):
    train_df = df_trainval.iloc[tr_idx].reset_index(drop=True)
    val_df   = df_trainval.iloc[va_idx].reset_index(drop=True)

    # Weighted sampler by label (1..7)
    counts  = train_df.label.value_counts().sort_index().values
    weights = 1.0 / (counts + 1e-6)
    samp_w  = train_df.label.apply(lambda l: weights[int(l)-1]).values
    sampler = WeightedRandomSampler(samp_w, len(samp_w), replacement=True)

    train_loader = make_loader(train_df, train_tf, sampler=sampler)
    val_loader   = make_loader(val_df,   val_tf,   shuffle=False)

    model = SiameseRegressor(EMB_DIM).to(device)
    opt   = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3, min_lr=1e-6)
    loss_fn = nn.MSELoss()

    best_mae, patience = 1e9, 0
    logs = {"tr_loss":[], "va_mae":[]}

    for epoch in range(1, EPOCHS+1):
        tr_loss, _, _, _, _, _ = run_epoch(model, train_loader, opt, loss_fn)
        va_loss, va_mae, va_std, va_tol, _, _ = run_epoch(model, val_loader, None, loss_fn)
        logs["tr_loss"].append(tr_loss)
        logs["va_mae"].append(va_mae)
        sch.step(va_mae)

        print(f"[Fold {fold}] Epoch {epoch:02d} | tr_loss {tr_loss:.4f} | va_MAE {va_mae:.4f} | ±0 {va_std:.1f}% | ±{TOL} {va_tol:.1f}% | patience {patience}")

        if va_mae < best_mae - 1e-3:
            best_mae, patience = va_mae, 0
            torch.save(model.state_dict(), f"/content/siam_best_fold{fold}.pth")
        else:
            patience += 1
            if patience >= PATIENCE:
                break

    model.load_state_dict(torch.load(f"/content/siam_best_fold{fold}.pth", map_location=device))
    model._train_logs = logs

    # Final fold eval
    _, vmae, vstd, vtol, _, _ = run_epoch(model, val_loader, None, loss_fn)
    mae_list.append(vmae); std_list.append(vstd); tol_list.append(vtol)
    fold_models.append(model)
    print(f"Fold {fold} DONE -> MAE {vmae:.3f} | ±0 {vstd:.1f}% | ±{TOL} {vtol:.1f}%")

print(f"\n5-fold avg MAE {np.mean(mae_list):.3f}±{np.std(mae_list):.3f}")
print(f"5-fold avg ±0 {np.mean(std_list):.1f}% | ±{TOL} {np.mean(tol_list):.1f}%")

# pick best fold
best_model = fold_models[np.argmin(mae_list)]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 113MB/s]


[Fold 1] Epoch 01 | tr_loss 2.7065 | va_MAE 0.9601 | ±0 30.6% | ±1 78.6% | patience 0
[Fold 1] Epoch 02 | tr_loss 1.7182 | va_MAE 0.8659 | ±0 34.1% | ±1 82.8% | patience 0
[Fold 1] Epoch 03 | tr_loss 1.5736 | va_MAE 0.8116 | ±0 40.6% | ±1 85.6% | patience 0
[Fold 1] Epoch 04 | tr_loss 1.2190 | va_MAE 0.8175 | ±0 42.9% | ±1 85.1% | patience 0
[Fold 1] Epoch 05 | tr_loss 1.2406 | va_MAE 0.7952 | ±0 42.5% | ±1 86.6% | patience 1
[Fold 1] Epoch 06 | tr_loss 1.1755 | va_MAE 0.7798 | ±0 42.0% | ±1 86.3% | patience 0
[Fold 1] Epoch 07 | tr_loss 1.1975 | va_MAE 0.7523 | ±0 42.0% | ±1 87.9% | patience 0
[Fold 1] Epoch 08 | tr_loss 1.0547 | va_MAE 0.7520 | ±0 46.0% | ±1 87.5% | patience 0
[Fold 1] Epoch 09 | tr_loss 1.1314 | va_MAE 0.7854 | ±0 42.0% | ±1 85.1% | patience 1
[Fold 1] Epoch 10 | tr_loss 0.9661 | va_MAE 0.7150 | ±0 46.6% | ±1 88.9% | patience 2
[Fold 1] Epoch 11 | tr_loss 1.0249 | va_MAE 0.7867 | ±0 44.8% | ±1 83.3% | patience 0
[Fold 1] Epoch 12 | tr_loss 0.9591 | va_MAE 0.7897 | ±

In [13]:
# %% (NEW) Plot learning curves per fold
import matplotlib.pyplot as plt

def plot_fold_curves(models, metric_name="MAE"):
    cols = 2
    rows = int(np.ceil(len(models)/cols))
    plt.figure(figsize=(12, 4*rows))
    for i, m in enumerate(models, 1):
        logs = getattr(m, "_train_logs", None)
        if logs is None:
            continue
        tr_loss = logs.get("tr_loss", [])
        va_mae  = logs.get("va_mae", [])
        ax = plt.subplot(rows, cols, i)
        ax.plot(tr_loss, label="Train Loss")
        ax.plot(va_mae,  label=f"Val {metric_name}")
        ax.set_title(f"Fold {i} Learning Curves")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss / MAE")
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

plot_fold_curves(fold_models, metric_name="MAE")



FINAL TEST SET (Siamese)
Test Loss (MSE): 0.7106
Test MAE       : 0.644
Test Acc (±0)  : 48.3%
Test Acc (±1): 92.0%


In [15]:
# %% 13) Held-out TEST evaluation + confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

test_loader = make_loader(df_test, val_tf, shuffle=False)
loss_fn = nn.MSELoss()

best_model.eval()
te_loss, te_mae, te_std, te_tol, te_preds, te_targs = run_epoch(best_model, test_loader, None, loss_fn)

print("="*50)
print("FINAL TEST SET (Siamese)")
print("="*50)
print(f"Test Loss (MSE): {te_loss:.4f}")
print(f"Test MAE       : {te_mae:.3f}")
print(f"Test Acc (±0)  : {te_std:.1f}%")
print(f"Test Acc (±{TOL}): {te_tol:.1f}%")

# ---- Confusion matrix on rounded predictions (clipped to 1..7)
pred_rounded = np.clip(np.rint(te_preds.numpy()), 1, 7).astype(int)
true_labels  = np.rint(te_targs.numpy()).astype(int)
labels = list(range(1, 8))
cm = confusion_matrix(true_labels, pred_rounded, labels=labels)

plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Test Set Confusion Matrix (Rounded Predictions)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


Saved to /content/siamese_best_model.pth


In [ ]:
# %% 14) Save best model
torch.save(best_model.state_dict(), "/content/siamese_best_model.pth")
print("Saved to /content/siamese_best_model.pth")

In [16]:
# %% Generate HTML files per shape for the *Siamese* notebook (child+reference, normalized)
import os, base64, cv2, numpy as np, torch, glob
from datetime import datetime

# --- sanity checks
assert 'best_model' in globals(), "best_model not found. Train/pick your Siamese model first."
assert 'val_tf' in globals(), "val_tf not found. Ensure your preprocessing (val_tf) is defined."
assert 'device' in globals(), "device not found. Ensure your notebook defined `device` (cpu/cuda)."
assert 'SHAPE_SOURCES' in globals(), "SHAPE_SOURCES not found."
assert 'df' in globals(), "df (merged labels with ref_path) not found. Run the merging cell before this."
assert 'SiameseRegressor' in globals() or hasattr(best_model, 'encode'), "best_model should be the Siamese network."

best_model.eval()

# Normalize SHAPE_SOURCES to {shape_name: folder_path}
shape_to_dir = {}
for k, v in SHAPE_SOURCES.items():
    if isinstance(v, (list, tuple)) and len(v) >= 2:
        shape_to_dir[k] = v[1]  # (score_file, img_dir)
    else:
        shape_to_dir[k] = v     # already a folder path

def img_to_data_uri(img_path: str) -> str:
    with open(img_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    ext = os.path.splitext(img_path)[1].lower()
    mime = "image/png" if ext == ".png" else ("image/jpeg" if ext in (".jpg", ".jpeg") else "image/png")
    return f"data:{mime};base64,{b64}"

def predict_scores_siamese(rows, batch_size=64):
    """
    rows: list of dicts with keys: img_path, ref_path
    Returns: list of raw predictions (floats in [1,7] if model bounds output)
    """
    preds = []
    with torch.no_grad():
        for i in range(0, len(rows), batch_size):
            chunk = rows[i:i+batch_size]
            child_tensors, ref_tensors = [], []
            for r in chunk:
                cimg = cv2.imread(r["img_path"], cv2.IMREAD_GRAYSCALE)
                rimg = cv2.imread(r["ref_path"], cv2.IMREAD_GRAYSCALE)
                if cimg is None: cimg = np.zeros((200,200), np.uint8)
                if rimg is None: rimg = np.zeros((200,200), np.uint8)
                ct = val_tf(image=cimg)["image"].float()  # SAME normalization as validation
                rt = val_tf(image=rimg)["image"].float()
                child_tensors.append(ct)
                ref_tensors.append(rt)
            xb_c = torch.stack(child_tensors, dim=0).to(device)
            xb_r = torch.stack(ref_tensors, dim=0).to(device)
            out = best_model(xb_c, xb_r).detach().cpu().numpy()
            preds.extend(out.tolist())
    return preds

def make_html_for_shape_siam(shape_name: str, folder: str):
    """
    Build HTML using rows coming from df (so we have labels and references),
    skipping any 'original*.png' entries as child images.
    """
    # Subset of df for this shape, with existing files and reference present
    sub = df[(df["shape_name"] == shape_name) & df["ref_path"].notna()].copy()
    sub = sub[sub["img_path"].apply(os.path.exists)]
    if sub.empty:
        print(f"⚠️ No rows for {shape_name}")
        return

    # Prepare rows (skip reference files as child images)
    rows = []
    for row in sub.itertuples(index=False):
        fname = os.path.basename(row.img_path).lower()
        if fname.startswith("original"):  # skip the reference file itself as a child sample
            continue
        rows.append({
            "child_id": str(row.child_id),
            "img_path": row.img_path,
            "ref_path": row.ref_path,
            "label":    int(row.label)
        })

    if not rows:
        print(f"⚠️ No child images for {shape_name} (after skipping originals).")
        return

    # Predict in mini-batches with proper normalization
    raw_preds = predict_scores_siamese(rows, batch_size=64)

    # Collect items with meta
    items = []
    for r, raw in zip(rows, raw_preds):
        rnd = int(np.clip(np.rint(raw), 1, 7))
        abs_err = abs(rnd - r["label"])
        items.append({
            "child_id": r["child_id"],
            "raw": float(raw),
            "rnd": rnd,
            "true": int(r["label"]),
            "abs_err": int(abs_err),
            "uri": img_to_data_uri(r["img_path"]),
            "ref_name": os.path.basename(r["ref_path"])
        })

    # Sort by predicted raw ascending
    items.sort(key=lambda x: x["raw"])

    # HTML
    ts = datetime.now().strftime("%Y-%m-%d %H:%M")
    html = [
        "<!doctype html><html><head><meta charset='utf-8'>",
        f"<title>{shape_name} predictions (Siamese)</title>",
        "<style>",
        "body{font-family:system-ui,-apple-system,Segoe UI,Roboto,Arial,sans-serif;margin:20px}",
        ".controls{margin:6px 0 14px 0;display:flex;gap:10px;align-items:center}",
        ".grid{display:grid;grid-template-columns:repeat(5,1fr);gap:12px}",
        ".card{border:1px solid #ddd;border-radius:12px;padding:10px;box-shadow:0 1px 3px rgba(0,0,0,.06)}",
        ".card.bad{border-color:#e57373}",
        ".img{display:flex;justify-content:center;align-items:center}",
        "img{max-width:100%;height:auto;border-radius:8px;background:#000}",
        ".meta{margin-top:8px;font-size:14px;line-height:1.3}",
        ".meta b{font-weight:600}",
        ".tag{display:inline-block;padding:2px 6px;border-radius:6px;font-size:12px;margin-left:6px;background:#eee}",
        ".bad .tag{background:#fde0e0}",
        "@media(max-width:1100px){.grid{grid-template-columns:repeat(4,1fr)}}",
        "@media(max-width:900px){.grid{grid-template-columns:repeat(3,1fr)}}",
        "@media(max-width:700px){.grid{grid-template-columns:repeat(2,1fr)}}",
        "@media(max-width:480px){.grid{grid-template-columns:1fr}}",
        "</style></head><body>",
        f"<h2>{shape_name} — Siamese predicted scores (ascending)</h2>",
        f"<div style='color:#666;font-size:13px'>Generated {ts} • Total {len(items)}</div>",
        "<div class='controls'>",
        "<button id='toggle'>Show only errors &gt; ±1</button>",
        "<span style='color:#666;font-size:13px'>Error computed on <b>rounded</b> predictions. Reference: original*.png</span>",
        "</div>",
        "<div class='grid' id='grid'>"
    ]

    for it in items:
        bad_cls = " bad" if it["abs_err"] > 1 else ""
        html += [
            f"<div class='card{bad_cls}' data-pred='{it['raw']:.4f}' "
            f"data-predr='{it['rnd']}' data-label='{it['true']}' data-abserr='{it['abs_err']}'>",
            f"  <div class='img'><img src='{it['uri']}' alt='{it['child_id']}'></div>",
            "  <div class='meta'>",
            f"    child_id: <b>{it['child_id']}</b><br>",
            f"    pred: <b>{it['rnd']}</b> <span class='tag'>raw {it['raw']:.2f}</span><br>",
            f"    label: <b>{it['true']}</b> <span class='tag'>|Δ|: {it['abs_err']}</span><br>",
            f"    ref: <span>{it['ref_name']}</span>",
            "  </div>",
            "</div>"
        ]

    html += [
        "</div>",
        "<script>",
        "const btn=document.getElementById('toggle');",
        "let showBad=false;",
        "btn.addEventListener('click',()=>{",
        "  showBad=!showBad;",
        "  btn.textContent=showBad?'Show all':'Show only errors > ±1';",
        "  document.querySelectorAll('.card').forEach(c=>{",
        "    const err=parseInt(c.dataset.abserr);",
        "    const isBad=!isNaN(err) && err>1;",
        "    c.style.display=(showBad? (isBad?'':'none') : '');",
        "  });",
        "});",
        "</script>",
        "</body></html>"
    ]

    out = f"/content/{shape_name}_siamese.html"
    with open(out, "w", encoding="utf-8") as f:
        f.write("\n".join(html))
    print(f"✅ Wrote {out} ({len(items)} images)")

# Run for all shapes
for shape, folder in shape_to_dir.items():
    if os.path.isdir(folder):
        make_html_for_shape_siam(shape, folder)
    else:
        print(f"⚠️ Skipping {shape}: folder not found -> {folder}")

print("Done. Files saved to /content/:", ", ".join([f"{s}_siamese.html" for s in shape_to_dir.keys()]))


✅ Wrote /content/shape7_siamese.html (893 images)
✅ Wrote /content/shape9_siamese.html (898 images)
✅ Wrote /content/shape14_siamese.html (890 images)
✅ Wrote /content/shape18_siamese.html (876 images)
Done. Files saved to /content/: shape7_siamese.html, shape9_siamese.html, shape14_siamese.html, shape18_siamese.html
